# Lesson 4 - Conversation State

Teach both current state patterns:

1. previous_response_id for a short response chain.
2. A durable Conversations API object for state reused across sessions, devices, or jobs.

The scenario reviews Fictional Beacon Books, compares it with Fictional Cedar Cycle Shop, and revises next steps without manually resending the entire exchange.

In [ ]:
import os

from openai import OpenAI

from harborlight_responses.config import api_key_available, resolve_model
from harborlight_responses.fixtures import FIXTURE_LABEL
from harborlight_responses.responses import policy_prompt
from harborlight_responses.services import get_policy

selection = resolve_model()
beacon = get_policy("FIC-HLA-1002")
cedar = get_policy("FIC-HLA-1003")
live_requested = os.getenv("HARBORLIGHT_LIVE") == "1"
live = live_requested and api_key_available()
instructions = (
    "Discuss only the supplied fictional Harborlight records. "
    "Keep next steps concise and do not provide insurance advice."
)
print("Selected model:", selection.model)
print("Mode:", "Live API" if live else "Demo Fixture")

In [ ]:
if live:
    client = OpenAI()
    first = client.responses.create(
        model=selection.model,
        instructions=instructions,
        input="Review Beacon Books:" + chr(10) + policy_prompt(beacon),
    )
    follow_up = client.responses.create(
        model=selection.model,
        instructions=instructions,
        previous_response_id=first.id,
        input="Compare it with Cedar Cycle Shop:" + chr(10) + policy_prompt(cedar),
    )
    print("Short chain, turn 1:")
    print(first.output_text)
    print()
    print("Short chain, turn 2:")
    print(follow_up.output_text)
else:
    print(FIXTURE_LABEL)
    print("Short-chain fixture: Beacon is +5.00%; Cedar is -3.00%.")
    print("No live response ID was created or stored.")

In [ ]:
if live:
    conversation = client.conversations.create()
    turn_1 = client.responses.create(
        model=selection.model,
        conversation=conversation.id,
        instructions=instructions,
        input="Review Beacon Books:" + chr(10) + policy_prompt(beacon),
    )
    turn_2 = client.responses.create(
        model=selection.model,
        conversation=conversation.id,
        instructions=instructions,
        input="Compare that renewal with Cedar Cycle Shop:" + chr(10) + policy_prompt(cedar),
    )
    turn_3 = client.responses.create(
        model=selection.model,
        conversation=conversation.id,
        instructions=instructions,
        input="Revise the next steps for the two fictional customers.",
    )
    print("Durable conversation, final turn:")
    print(turn_3.output_text)
else:
    print(FIXTURE_LABEL)
    print(
        "Durable-state fixture: follow up with Beacon about its increase and "
        "confirm Cedar's lower proposal. No Conversation object was created."
    )

## Choosing and paying for state

Use previous_response_id for a short linear chain. Use a Conversation object when state must persist across sessions, devices, or jobs. Response objects are stored for 30 days by default unless store=False; items attached to Conversations persist beyond that response TTL. Even with previous_response_id, prior input tokens in the chain are billed again.

Instructions apply only to the current response request, so this lesson repeats the same instructions on each turn. Never commit live response or conversation identifiers. Fixture mode demonstrates expected behavior but creates no remote state.